# Assistant multi-agent pour la sélection intelligente de portefeuilles financiers

**Projet de Machine Learning** – Notebook principal (code + résultats)

Ce notebook implémente de bout en bout un système multi-agent qui, à partir du profil d'un investisseur et de l'état des marchés financiers, propose une allocation de portefeuille diversifiée, contrôlée en risque et expliquée.

| Agent | Rôle | Technique ML / quantitative | Données réelles |
|---|---|---|---|
| **Agent Profil** | Estimer la tolérance au risque du client | Classification supervisée multi-classes | Survey of Consumer Finances 2022 (Réserve fédérale US) |
| **Agent Régime** | Détecter l'état du marché (Calme / Neutre / Stress) | Apprentissage non supervisé (mélange gaussien) | FRED + Yahoo Finance |
| **Agent Prévision** | Prédire quels actifs vont surperformer le mois suivant | Classification supervisée binaire (LightGBM, XGBoost, RF, MLP…) | Yahoo Finance (25 ETF) + FRED |
| **Agent Risque** | Estimer la covariance, contrôler VaR / volatilité | Estimateur de Ledoit-Wolf | Yahoo Finance |
| **Agent Allocation** | Construire le portefeuille optimal | Optimisation moyenne-variance sous contraintes | – |
| **Agent Explication** | Justifier la recommandation | Synthèse des sorties + importance des variables | – |

**Plan du notebook**
0. Configuration
1. Collecte des données réelles (Yahoo Finance, FRED, SCF)
2. Prétraitement des données de marché
3. Analyse exploratoire des marchés
4. Agent Profil : données SCF, prétraitement, baseline, modèles, interprétabilité
5. Agent Prévision : ingénierie des variables, découpage temporel purgé, baseline, modèles, interprétabilité
6. Agent Régime : mélange gaussien
7. Orchestration multi-agent : démonstration
8. Backtest walk-forward et comparaison aux stratégies de référence
9. Sauvegarde des artefacts (pour le déploiement Streamlit)
10. Synthèse automatique des résultats et conclusion

> ⚠️ **Prévention des fuites de données** : les jeux d'entraînement, de validation et de test sont séparés **avant** tout prétraitement ajusté sur les données. Tous les imputeurs, normalisations et écrêtages sont encapsulés dans des `Pipeline` scikit-learn et appris uniquement sur l'entraînement.

## 0. Configuration de l'environnement

Installation (une seule fois) : `pip install -r requirements.txt` depuis la racine du projet.

`QUICK_RUN = True` réduit les grilles d'hyperparamètres pour un test rapide ; laisser `False` pour les résultats finaux.

In [ ]:
import sys
import json
import time
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import (train_test_split, StratifiedGroupKFold,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

from src.config import *
from src import data_download as dl
from src.preprocessing import (clean_prices, compute_returns, build_macro_features,
                               risk_free_daily, prepare_scf, Winsorizer, SignedLog,
                               MONEY_FEATURES)
from src.features import (compute_asset_features, compute_market_features, sample_dates,
                          temporal_split, PurgedTimeSeriesSplit, information_coefficient,
                          newey_west_tstat,
                          long_short_spread, ASSET_NUM_FEATURES, MACRO_FEATURES,
                          CAT_FEATURES, REGIME_FEATURES)
from src.agents import PortfolioOrchestrator, AllocationAgent, RiskAgent
from src.questionnaire import QUESTIONS, score_questionnaire
from src.backtest import (perf_metrics, simulate, equal_weight, sixty_forty, min_variance,
                          markowitz_historical, rebalance_dates, vol_targeted_equal_weight)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (12, 5), "axes.titleweight": "bold"})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.random.seed(RANDOM_STATE)

QUICK_RUN = False   # True = test rapide uniquement (grilles réduites) – NE PAS utiliser pour les résultats finaux
N_JOBS = -1
RESULTS = {}   # résultats collectés pour la synthèse finale


def savefig(name: str):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


print(f"Racine du projet : {ROOT}")
print(f"LightGBM : {HAS_LGBM} | XGBoost : {HAS_XGB} | SHAP : {HAS_SHAP}")

## 1. Collecte des données réelles

### 1.1 Sources

| Source | Contenu | Fréquence | Accès |
|---|---|---|---|
| **Yahoo Finance** (`yfinance`) | Cours de clôture ajustés des dividendes et divisions + volumes de 25 actifs cotés (ETF actions, secteurs, obligations, immobilier, matières premières, Bitcoin) | Quotidienne, 2010 → aujourd'hui | API publique |
| **FRED** – Federal Reserve Bank of St. Louis | Taux 2 ans / 10 ans / 3 mois, pente de courbe, spread de crédit Baa, VIX, inflation (CPI), chômage, production industrielle | Quotidienne / mensuelle | CSV public, sans clé |
| **Survey of Consumer Finances 2022** – Board of Governors of the Federal Reserve System | Enquête triennale sur ~4 600 ménages américains : âge, revenu, patrimoine, dettes, éducation, connaissances financières **et tolérance au risque déclarée** | Coupe transversale | Fichier « Summary Extract Public Data » |

Les données brutes sont mises en cache dans `data/raw/` : relancer le notebook ne retélécharge rien (reproductibilité). Pour rafraîchir : `dl.download_all(force=True)`.

> Alternative en cas de blocage de Yahoo Finance : le jeu Kaggle « Huge Stock Market Dataset » ou un export manuel CSV des mêmes ETF peut être placé dans `data/raw/yahoo_close_raw.csv` (format : date en index, un ticker par colonne).

In [ ]:
pd.DataFrame([(t, n, c) for t, (n, c) in UNIVERSE.items()],
             columns=["Ticker", "Libellé", "Classe d'actifs"]).set_index("Ticker")

In [ ]:
close_raw, volume_raw = dl.download_yahoo()
fred_raw = dl.download_fred()
scf_raw = dl.download_scf()

print("Cours Yahoo  :", close_raw.shape, close_raw.index.min().date(), "→", close_raw.index.max().date())
print("Séries FRED  :", fred_raw.shape)
print("SCF          :", scf_raw.shape)
display(close_raw.tail())

## 2. Prétraitement des données de marché

### 2.1 Diagnostic de qualité des données brutes
Problèmes typiques des données financières réelles :
- **calendriers différents** : le Bitcoin cote 7 j/7, les ETF uniquement les jours ouvrés NYSE ;
- **historiques de longueurs différentes** (Bitcoin disponible sur Yahoo à partir de septembre 2014) ;
- **valeurs manquantes** ponctuelles (jours fériés, suspensions, trous de l'API) ;
- **valeurs aberrantes** (pics isolés dus à des erreurs de cotation).

In [ ]:
first = close_raw.apply(lambda s: s.first_valid_index())
quality = pd.DataFrame({
    "Première cotation": first.dt.date,
    "Dernière cotation": close_raw.apply(lambda s: s.last_valid_index()).dt.date,
    "Nb observations": close_raw.count(),
    "% manquants après 1re cotation": [close_raw.loc[first[c]:, c].isna().mean() * 100 for c in close_raw],
    "Jours à volume nul": (volume_raw == 0).sum(),
})
quality

In [ ]:
prices, clean_report = clean_prices(close_raw)
for k in ("dates_brutes", "dates_calendrier_NYSE", "prix_non_positifs", "pics_aberrants"):
    print(f"{k:<28}: {clean_report[k]}")
miss = pd.DataFrame({"avant ffill": clean_report["valeurs_manquantes_avant_ffill"],
                     "après ffill": clean_report["valeurs_manquantes_apres_ffill"]})
display(miss.T)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(prices.isna().T.astype(int).iloc[:, ::5], cbar=False, cmap="Greys", ax=ax)
ax.set_title("Disponibilité des données après nettoyage (noir = manquant)")
ax.set_xticks([])
savefig("01_disponibilite_donnees")
plt.show()

### 2.2 Rendements et valeurs extrêmes
On travaille sur les **rendements** (stationnaires) et non sur les prix (non stationnaires). Les rendements extrêmes réels (krach COVID de mars 2020, par exemple) sont **conservés** : ce sont de vraies informations de marché et non des erreurs. Seuls les pics isolés aller-retour ont été retirés à l'étape précédente.

In [ ]:
returns = compute_returns(prices)
z = (returns - returns.mean()) / returns.std()
extremes = (z.abs() > 6).sum().sort_values(ascending=False)
print("Nombre de rendements à plus de 6 écarts-types (conservés) :")
display(extremes[extremes > 0].to_frame("nb").T)

### 2.3 Variables macroéconomiques sans fuite temporelle
Une valeur macroéconomique n'est utilisable qu'à sa **date de publication**, pas à sa date de référence :
- séries quotidiennes (taux, VIX) décalées d'un jour ouvré ;
- séries mensuelles (inflation, chômage) disponibles seulement le 15 du mois suivant.

In [ ]:
macro = build_macro_features(fred_raw, prices.index)
rf_daily = risk_free_daily(macro)
display(macro.describe().T)
print("Valeurs manquantes résiduelles :")
display(macro.isna().sum().to_frame("nb").T)

prices.to_csv(DATA_PROC / "prices_clean.csv")
returns.to_csv(DATA_PROC / "returns_daily.csv")
macro.to_csv(DATA_PROC / "macro_features.csv")
print("Données nettoyées enregistrées dans", DATA_PROC)

## 3. Analyse exploratoire des marchés

Pour ne pas biaiser nos choix de modélisation par des informations de la période de test, les statistiques qui **guident la modélisation** sont calculées sur la période d'entraînement. Les graphiques de contexte couvrent toute la période.

In [ ]:
train_mask = returns.index <= TRAIN_END
R_tr = returns[train_mask]


def describe_returns(R: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "Classe": pd.Series(ASSET_CLASS),
        "Rdt annualisé": R.mean() * 252,
        "Vol annualisée": R.std() * np.sqrt(252),
        "Asymétrie": R.skew(),
        "Kurtosis (excès)": R.kurt(),
        "Max drawdown": ((1 + R.fillna(0)).cumprod() / (1 + R.fillna(0)).cumprod().cummax() - 1).min(),
        "p-val Jarque-Bera": [stats.jarque_bera(R[c].dropna()).pvalue for c in R],
    })
    out["Sharpe (rf=0)"] = out["Rdt annualisé"] / out["Vol annualisée"]
    return out.loc[R.columns]


stats_train = describe_returns(R_tr)
stats_train.to_csv(RESULTS_DIR / "stats_actifs_train.csv")
stats_train.sort_values("Classe")

**Lecture** : les rendements journaliers présentent tous une kurtosis largement positive (queues épaisses) et la normalité est rejetée (p-valeurs de Jarque-Bera ≈ 0). Cela justifie (i) des mesures de risque historiques (CVaR) en plus de la volatilité, (ii) des modèles non linéaires robustes aux valeurs extrêmes (arbres de décision).

In [ ]:
show = ["SPY", "EFA", "EEM", "TLT", "IEF", "HYG", "GLD", "VNQ", "DBC"]
norm = prices[show].dropna().pipe(lambda d: d / d.iloc[0] * 100)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
norm.plot(ax=axes[0], lw=1.2)
axes[0].set_title("Performance cumulée (base 100)")
axes[0].set_ylabel("Indice")
(prices["BTC-USD"].dropna() / prices["BTC-USD"].dropna().iloc[0] * 100).plot(ax=axes[1], logy=True, color="darkorange")
axes[1].set_title("Bitcoin (échelle logarithmique)")
for d, lab in [("2020-03-23", "COVID"), ("2022-10-12", "Creux 2022")]:
    for ax in axes:
        ax.axvline(pd.Timestamp(d), color="red", ls="--", alpha=.5)
savefig("02_performances_cumulees")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(R_tr.corr(), cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={"shrink": .7})
ax.set_title("Corrélation des rendements quotidiens (période d'entraînement)")
savefig("03_correlations")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
roll_corr = returns["SPY"].rolling(126).corr(returns["TLT"])
roll_corr.plot(ax=axes[0], color="purple")
axes[0].axhline(0, color="k", lw=.8)
axes[0].set_title("Corrélation glissante 6 mois actions / obligations longues")

r = returns["SPY"].dropna()
sns.histplot(r, bins=150, stat="density", ax=axes[1], color="steelblue")
xx = np.linspace(r.min(), r.max(), 300)
axes[1].plot(xx, stats.norm.pdf(xx, r.mean(), r.std()), "r", label="Loi normale")
axes[1].set_xlim(-0.06, 0.06)
axes[1].legend()
axes[1].set_title("Distribution des rendements du S&P 500")

lags = range(1, 31)
axes[2].bar([l - .2 for l in lags], [r.autocorr(l) for l in lags], width=.4, label="rendements")
axes[2].bar([l + .2 for l in lags], [r.abs().autocorr(l) for l in lags], width=.4, label="|rendements|")
axes[2].set_title("Autocorrélations : regroupement de volatilité")
axes[2].legend()
savefig("04_faits_stylises")
plt.show()

**Faits stylisés observés** : (1) la corrélation actions/obligations, négative pendant la décennie 2010, devient positive en 2022 (choc d'inflation) : la diversification n'est pas stable et doit être réestimée dynamiquement ; (2) queues épaisses ; (3) les rendements sont peu autocorrélés mais leurs valeurs absolues le sont fortement (regroupement de volatilité) : la volatilité récente est prévisible, d'où l'intérêt de l'agent Régime et d'une covariance glissante.

In [ ]:
cols = [c for c in ["VIX", "T10Y2Y", "BAA10Y", "CPI_yoy", "UNRATE"] if c in macro]
macro[cols].plot(subplots=True, figsize=(13, 9), lw=1, title="Indicateurs macro-financiers (FRED)")
plt.tight_layout()
savefig("05_macro")
plt.show()

## 4. Agent Profil investisseur – classification de la tolérance au risque

### 4.1 Problématique et fondements
À partir de caractéristiques observables d'un client (âge, revenus, patrimoine, endettement, éducation, connaissances financières, comportements d'épargne et de crédit, façon de prendre ses décisions d'investissement), prédire sa tolérance au risque.

La cible provient de la question du SCF sur la disposition à prendre des risques financiers (`NOFINRISK`, `YESFINRISK`). C'est une donnée **réelle** déclarée par les ménages.

**Choix de formulation (diagnostic de la version 1)** : la formulation à 3 classes (Prudent / Équilibré / Dynamique) plafonnait à un F1 macro d'environ 0,50 et une AUC de 0,74, quel que soit le modèle. Le diagnostic montre que la classe « Dynamique » ne représente que **5 %** des ménages et n'est pas séparable d'« Équilibré » avec des variables socio-économiques. Même en ajoutant (à titre de test) des variables qui constitueraient une fuite, comme la détention d'actions, le F1 macro ne dépasse pas 0,50 : la limite vient de la cible, pas des modèles. En revanche, détecter **l'aversion au risque** (Prudent vs Non prudent) est bien appris (AUC ≈ 0,84 lors du diagnostic).

Le système adopte donc une démarche en deux temps, proche des questionnaires réglementaires (MiFID II) :
1. **modèle ML** : probabilité d'aversion au risque (classification binaire, seuil de décision optimisé sur la validation) ;
2. **règle explicite** : si le client n'est pas prudent, il est classé « Dynamique » seulement s'il déclare un horizon d'au moins 10 ans **et** accepte une perte temporaire d'au moins 20 %, sinon « Équilibré ».

Le paramètre `PROFILE_TARGET_MODE` de `src/config.py` permet de revenir à la formulation à 3 classes pour comparaison.

**Tolérance déclarée et capacité estimée** : dans l'assistant, le client remplit un **questionnaire noté de 7 questions** (horizon, objectif, réaction à une baisse de 20 %, perte acceptable, expérience, part du patrimoine placée, stabilité des revenus ; `src/questionnaire.py`). Ce questionnaire mesure sa **tolérance au risque déclarée**. Le modèle ML entraîné ici estime sa **capacité à supporter des pertes** à partir de sa situation financière. Ce sont les deux notions que distingue la réglementation (MiFID II). Le profil retenu est **le plus prudent des deux**, et toute divergence est signalée au conseiller. Le barème du questionnaire est fixé a priori, car aucune base publique ne contient de réponses à un tel questionnaire.

**Spécificité méthodologique du SCF – risque de fuite** : les valeurs manquantes y sont imputées 5 fois (*multiple imputation*) et chaque ménage apparaît sur **5 lignes quasi identiques** (`YY1` = identifiant ménage). Un découpage aléatoire ligne par ligne placerait des copies d'un même ménage dans l'entraînement et le test et gonflerait artificiellement les scores. Nous découpons donc **par ménage** (groupes).

In [ ]:
X_p, y_p, groups_p, w_p, scf_report = prepare_scf(scf_raw)
num_p, cat_p = scf_report["variables_numeriques"], scf_report["variables_categorielles"]
print(json.dumps({k: v for k, v in scf_report.items() if k != "valeurs_manquantes"},
                 indent=1, ensure_ascii=False, default=str))
display(X_p.head())

### 4.2 Découpage train / validation / test (70 / 15 / 15) **avant** tout prétraitement
Stratifié sur la classe, et par ménage.

In [ ]:
hh = pd.DataFrame({"g": groups_p.values, "y": y_p.astype(str).values}).groupby("g")["y"] \
    .agg(lambda s: s.mode().iloc[0])
hh_train, hh_tmp = train_test_split(hh.index, test_size=0.30, stratify=hh.values,
                                    random_state=RANDOM_STATE)
hh_val, hh_test = train_test_split(hh_tmp, test_size=0.50, stratify=hh.loc[hh_tmp].values,
                                   random_state=RANDOM_STATE)
m_tr, m_va, m_te = (groups_p.isin(h).values for h in (hh_train, hh_val, hh_test))
assert not (set(hh_train) & set(hh_val) or set(hh_train) & set(hh_test) or set(hh_val) & set(hh_test))

CLS_TO_INT = {c: i for i, c in enumerate(PROFILE_ML_CLASSES)}
BINARY = len(PROFILE_ML_CLASSES) == 2
y_int = y_p.astype(str).map(CLS_TO_INT).astype(int)
Xp_tr, Xp_va, Xp_te = X_p[m_tr], X_p[m_va], X_p[m_te]
yp_tr, yp_va, yp_te = y_int[m_tr], y_int[m_va], y_int[m_te]
gp_tr = groups_p[m_tr]

split_tab = pd.DataFrame({
    "lignes": [m_tr.sum(), m_va.sum(), m_te.sum()],
    "ménages": [len(hh_train), len(hh_val), len(hh_test)],
    **{c: [(y.values == i).mean() for y in (yp_tr, yp_va, yp_te)] for c, i in CLS_TO_INT.items()},
}, index=["train", "validation", "test"])
split_tab

### 4.3 Analyse exploratoire (sur l'entraînement uniquement)

In [ ]:
eda = Xp_tr.copy()
eda["profil"] = yp_tr.map({v: k for k, v in CLS_TO_INT.items()})
fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
eda["profil"].value_counts().reindex(PROFILE_ML_CLASSES).plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452", "#C44E52"][:len(PROFILE_ML_CLASSES)])
axes[0].set_title("Répartition de la cible (déséquilibrée)")
sns.boxplot(data=eda, x="profil", y="AGE", order=PROFILE_ML_CLASSES, ax=axes[1])
axes[1].set_title("Âge par profil")
eda["log_revenu"] = np.log1p(eda["INCOME"].clip(lower=0))
sns.boxplot(data=eda, x="profil", y="log_revenu", order=PROFILE_ML_CLASSES, ax=axes[2])
axes[2].set_title("log(1 + revenu) par profil")
if "KNOWL" in eda:
    pd.crosstab(eda["KNOWL"], eda["profil"], normalize="index")[PROFILE_ML_CLASSES] \
        .plot.bar(stacked=True, ax=axes[3], legend=False)
    axes[3].set_title("Profil selon la connaissance financière (0-10)")
plt.tight_layout()
savefig("06_eda_profils")
plt.show()

if "EDCL" in eda:
    display(pd.crosstab(eda["EDCL"], eda["profil"], normalize="index")[PROFILE_ML_CLASSES]
            .rename_axis("Niveau d'éducation (1=<bac … 4=diplôme univ.)"))
display(Xp_tr[num_p].describe().T)

**Lecture** : les classes sont déséquilibrées → on utilise des **poids de classes équilibrés** et le **F1 macro** comme métrique principale (l'accuracy serait trompeuse : prédire toujours la classe majoritaire donne déjà une accuracy élevée). Les montants (revenu, patrimoine, dettes) sont très asymétriques → transformation logarithmique signée.

### 4.4 Pipeline de prétraitement
- **montants** : imputation par la médiane → log signé → standardisation ;
- **autres numériques** : imputation médiane → écrêtage aux quantiles 1 %/99 % (appris sur train) → standardisation ;
- **catégorielles** : imputation par la modalité la plus fréquente → encodage one-hot.

In [ ]:
money_p = [c for c in MONEY_FEATURES if c in num_p]
other_p = [c for c in num_p if c not in money_p]
pre_profile = ColumnTransformer([
    ("montants", Pipeline([("imp", SimpleImputer(strategy="median")), ("log", SignedLog()),
                           ("sc", StandardScaler())]), money_p),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("win", Winsorizer(0.01, 0.99)),
                      ("sc", StandardScaler())]), other_p),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_p),
])

### 4.5 Modèles comparés
| Modèle | Rôle | Idée |
|---|---|---|
| Classe majoritaire / tirage stratifié | **Baselines** | Niveau de performance « sans apprentissage » |
| Régression logistique multinomiale | Modèle linéaire de référence | Probabilités par softmax d'une combinaison linéaire |
| Forêt aléatoire | Ensemble par *bagging* | Moyenne d'arbres décorrélés → réduit la variance |
| LightGBM / XGBoost (ou HistGradientBoosting) | *Boosting* de gradient | Arbres ajoutés séquentiellement pour corriger les erreurs résiduelles |
| Perceptron multicouche | Réseau de neurones | Non-linéarités apprises par rétropropagation |

Hyperparamètres choisis par **validation croisée stratifiée par groupes (5 plis)** sur l'entraînement, critère F1 macro. Sélection finale sur la validation, évaluation unique sur le test.

In [ ]:
def profile_models():
    m = {
        "Baseline – classe majoritaire": (DummyClassifier(strategy="most_frequent"), {}),
        "Baseline – tirage stratifié": (DummyClassifier(strategy="stratified", random_state=RANDOM_STATE), {}),
        "Régression logistique": (LogisticRegression(max_iter=3000, class_weight="balanced"),
                                  {"clf__C": [0.01, 0.1, 1, 10]}),
        "Forêt aléatoire": (RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                   n_jobs=1, random_state=RANDOM_STATE),
                            {"clf__max_depth": [6, 12, None], "clf__min_samples_leaf": [5, 20]}),
        "MLP": (MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=500,
                              random_state=RANDOM_STATE),
                {"clf__alpha": [1e-4, 1e-2], "clf__hidden_layer_sizes": [(64, 32), (128, 64)]}),
    }
    if HAS_LGBM:
        m["LightGBM"] = (lgb.LGBMClassifier(n_estimators=400, class_weight="balanced", subsample=0.8,
                                            subsample_freq=1, colsample_bytree=0.8, verbose=-1,
                                            n_jobs=1, random_state=RANDOM_STATE),
                         {"clf__learning_rate": [0.03, 0.1], "clf__num_leaves": [15, 31],
                          "clf__min_child_samples": [20, 50]})
    else:
        m["HistGradientBoosting"] = (HistGradientBoostingClassifier(class_weight="balanced",
                                                                    random_state=RANDOM_STATE),
                                     {"clf__learning_rate": [0.03, 0.1], "clf__max_leaf_nodes": [15, 31]})
    if HAS_XGB:
        m["XGBoost"] = (xgb.XGBClassifier(n_estimators=400, objective="binary:logistic" if BINARY else "multi:softprob", subsample=0.8,
                                          colsample_bytree=0.8, tree_method="hist", eval_metric="logloss" if BINARY else "mlogloss", n_jobs=1,
                                          random_state=RANDOM_STATE),
                        {"clf__learning_rate": [0.03, 0.1], "clf__max_depth": [3, 6]})
    if QUICK_RUN:
        m = {k: (e, {p: v[:1] for p, v in g.items()}) for k, (e, g) in m.items()}
    return m


def predict_with_threshold(proba, threshold=None):
    """Binaire : classe 0 (Prudent) si P(Prudent) ≥ seuil. Multi-classes : argmax."""
    if proba.shape[1] == 2 and threshold is not None:
        return np.where(proba[:, 0] >= threshold, 0, 1)
    return proba.argmax(1)


def eval_multiclass(model, X, y, threshold=None, sample_weight=None) -> dict:
    proba = model.predict_proba(X)
    pred = predict_with_threshold(proba, threshold)
    sw = sample_weight
    auc = roc_auc_score(y, proba[:, 1], sample_weight=sw) if proba.shape[1] == 2 else \
        roc_auc_score(y, proba, multi_class="ovr", average="macro", sample_weight=sw)
    out = {"Accuracy": accuracy_score(y, pred, sample_weight=sw),
           "Balanced acc.": balanced_accuracy_score(y, pred, sample_weight=sw),
           "F1 macro": f1_score(y, pred, average="macro", sample_weight=sw), "ROC-AUC": auc}
    if proba.shape[1] == 2:   # rappel de la classe « Prudent » (codée 0)
        out["Rappel Prudent"] = ((pred == 0) & (np.asarray(y) == 0)).sum() / max((np.asarray(y) == 0).sum(), 1) \
            if sw is None else np.sum(sw * ((pred == 0) & (np.asarray(y) == 0))) / np.sum(sw * (np.asarray(y) == 0))
    return out


cv_p = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
profile_fitted, rows = {}, []
for name, (est, grid) in profile_models().items():
    t0 = time.time()
    pipe = Pipeline([("pre", pre_profile), ("clf", est)])
    fit_params = {"groups": gp_tr}
    if name == "XGBoost":
        fit_params["clf__sample_weight"] = compute_sample_weight("balanced", yp_tr)
    gs = GridSearchCV(pipe, grid or {"clf__strategy": [est.strategy]}, cv=cv_p, scoring="f1_macro",
                      n_jobs=N_JOBS, refit=True)
    gs.fit(Xp_tr, yp_tr, **fit_params)
    profile_fitted[name] = gs
    rows.append({"Modèle": name, "F1 macro CV (train)": gs.best_score_,
                 **eval_multiclass(gs.best_estimator_, Xp_va, yp_va),
                 "Durée (s)": time.time() - t0,
                 "Meilleurs hyperparamètres": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}})
    print(f"✓ {name:<32} F1 macro val = {rows[-1]['F1 macro']:.3f}")

profile_comparison = pd.DataFrame(rows).set_index("Modèle").sort_values("F1 macro", ascending=False)
profile_comparison.to_csv(RESULTS_DIR / "agent_profil_comparaison_validation.csv")
RESULTS["profil_validation"] = profile_comparison.drop(columns="Meilleurs hyperparamètres")
profile_comparison

In [ ]:
best_profile_name = profile_comparison.drop(index=[i for i in profile_comparison.index if "Baseline" in i]).index[0]
print("Modèle retenu pour l'agent Profil :", best_profile_name)

# Seuil de décision choisi sur la VALIDATION (modèle entraîné sur le train seul) – jamais sur le test.
# Règle métier : ne pas repérer un client prudent est l'erreur la plus grave. On exige donc un rappel
# d'au moins PRUDENT_RECALL_MIN sur la classe « Prudent », puis on maximise l'accuracy équilibrée.
THRESH = 0.5
if BINARY:
    pv = profile_fitted[best_profile_name].best_estimator_.predict_proba(Xp_va)
    grid_t = np.round(np.arange(0.10, 0.81, 0.01), 2)
    yv = np.asarray(yp_va)
    bal_t = np.array([balanced_accuracy_score(yv, predict_with_threshold(pv, t)) for t in grid_t])
    rec_t = np.array([((predict_with_threshold(pv, t) == 0) & (yv == 0)).sum() / (yv == 0).sum() for t in grid_t])
    ok = rec_t >= PRUDENT_RECALL_MIN
    THRESH = float(grid_t[ok][np.argmax(bal_t[ok])]) if ok.any() else float(grid_t[np.argmax(rec_t)])
    plt.figure(figsize=(7, 3.8))
    plt.plot(grid_t, bal_t, label="Accuracy équilibrée")
    plt.plot(grid_t, rec_t, label="Rappel « Prudent »")
    plt.axhline(PRUDENT_RECALL_MIN, color="grey", ls=":", label=f"Rappel minimal exigé ({PRUDENT_RECALL_MIN:.0%})")
    plt.axvline(THRESH, color="r", ls="--")
    plt.xlabel("Seuil sur P(Prudent)")
    plt.legend()
    plt.title(f"Choix du seuil de décision (validation) : {THRESH:.2f}")
    savefig("07b_profil_seuil")
    plt.show()
print(f"Seuil retenu : {THRESH:.2f}")

# Ré-entraînement sur train + validation avec les meilleurs hyperparamètres, puis évaluation UNIQUE sur le test
best_profile = clone(profile_fitted[best_profile_name].best_estimator_)
X_trva, y_trva = pd.concat([Xp_tr, Xp_va]), pd.concat([yp_tr, yp_va])
fp = {"clf__sample_weight": compute_sample_weight("balanced", y_trva)} if best_profile_name == "XGBoost" else {}
best_profile.fit(X_trva, y_trva, **fp)

test_rows = {best_profile_name: eval_multiclass(best_profile, Xp_te, yp_te)}
if BINARY:
    test_rows[f"{best_profile_name} + seuil {THRESH:.2f}"] = eval_multiclass(best_profile, Xp_te, yp_te, THRESH)
    # Métriques pondérées par les poids d'enquête (représentativité de la population US)
    test_rows[f"{best_profile_name} + seuil – pondéré WGT"] = eval_multiclass(
        best_profile, Xp_te, yp_te, THRESH, sample_weight=w_p[m_te].values)
for b in [i for i in profile_comparison.index if "Baseline" in i]:
    test_rows[b] = eval_multiclass(profile_fitted[b].best_estimator_, Xp_te, yp_te)
profile_test = pd.DataFrame(test_rows).T
profile_test.to_csv(RESULTS_DIR / "agent_profil_test.csv")
RESULTS["profil_test"] = profile_test
display(profile_test)
pred_te_p = predict_with_threshold(best_profile.predict_proba(Xp_te), THRESH if BINARY else None)
print(classification_report(yp_te, pred_te_p, target_names=PROFILE_ML_CLASSES, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(yp_te, pred_te_p, display_labels=PROFILE_ML_CLASSES,
                                        normalize="true", cmap="Blues", ax=axes[0], values_format=".2f")
axes[0].set_title(f"Matrice de confusion normalisée – {best_profile_name} (test)")
proba_te = best_profile.predict_proba(Xp_te)
for i, c in enumerate(PROFILE_ML_CLASSES[:1] if BINARY else PROFILE_ML_CLASSES):
    RocCurveDisplay.from_predictions((yp_te == i).astype(int), proba_te[:, i], name=c, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("Courbes ROC un-contre-tous (test)")
plt.tight_layout()
savefig("07_profil_confusion_roc")
plt.show()

### 4.6 Interprétabilité de l'agent Profil
1. **Importance par permutation** (agnostique au modèle) : baisse du F1 macro quand on permute aléatoirement une variable sur le test.
2. **Valeurs SHAP** (théorie des jeux coopératifs, valeurs de Shapley) : contribution de chaque variable à chaque prédiction individuelle.
3. **Analyse détaillée des erreurs** : performance par tranche d'âge.

In [ ]:
pi = permutation_importance(best_profile, Xp_te, yp_te, scoring="f1_macro", n_repeats=10 if not QUICK_RUN else 3,
                            random_state=RANDOM_STATE, n_jobs=N_JOBS)
pi_profile = pd.DataFrame({"importance": pi.importances_mean, "écart-type": pi.importances_std},
                          index=Xp_te.columns).sort_values("importance")
pi_profile.plot.barh(y="importance", xerr="écart-type", legend=False, figsize=(8, 6), color="teal")
plt.title("Importance par permutation – agent Profil (baisse du F1 macro)")
savefig("08_profil_permutation")
plt.show()
pi_profile.sort_values("importance", ascending=False).to_csv(RESULTS_DIR / "agent_profil_importance_permutation.csv")

In [ ]:
def shap_beeswarm(pipe, X, title, fname, class_idx=None, max_n=2000):
    """Graphique SHAP pour un pipeline (prétraitement + modèle à base d'arbres)."""
    if not HAS_SHAP:
        print("SHAP non installé – seule l'importance par permutation est présentée.")
        return None
    pre, mdl = pipe.named_steps["pre"], pipe.named_steps["clf"]
    Xs = X.sample(min(max_n, len(X)), random_state=RANDOM_STATE)
    Xt = pd.DataFrame(np.asarray(pre.transform(Xs), dtype=float), columns=pre.get_feature_names_out())
    try:
        explainer = shap.TreeExplainer(mdl)
    except Exception:
        background = shap.sample(Xt, 100, random_state=RANDOM_STATE)
        explainer = shap.Explainer(lambda a: mdl.predict_proba(a), background)
    sv = explainer(Xt)
    if sv.values.ndim == 3:
        sv = sv[:, :, class_idx if class_idx is not None else -1]
    plt.figure()
    shap.plots.beeswarm(sv, max_display=15, show=False)
    plt.title(title)
    savefig(fname)
    plt.show()
    return sv


tree_names = [n for n in profile_comparison.index if n in ("LightGBM", "XGBoost", "Forêt aléatoire", "HistGradientBoosting")]
shap_model_name = best_profile_name if best_profile_name in tree_names else tree_names[0]
shap_pipe = best_profile if shap_model_name == best_profile_name else profile_fitted[shap_model_name].best_estimator_
for cls in (["Non prudent"] if BINARY else ["Dynamique", "Prudent"]):
    shap_beeswarm(shap_pipe, Xp_te, f"SHAP – probabilité « {cls} » ({shap_model_name})",
                  f"09_profil_shap_{cls.replace(' ', '_')}", class_idx=CLS_TO_INT[cls])

In [ ]:
err = pd.DataFrame({"age": Xp_te["AGE"], "vrai": yp_te, "prédit": pred_te_p})
err["tranche d'âge"] = pd.cut(err["age"], [0, 35, 50, 65, 120], labels=["<35", "35-50", "50-65", "65+"])
err_tab = err.groupby("tranche d'âge", observed=True).apply(
    lambda g: pd.Series({"n": len(g), "accuracy": (g["vrai"] == g["prédit"]).mean(),
                         "F1 macro": f1_score(g["vrai"], g["prédit"], average="macro")}))
display(err_tab)
print("Confusions les plus fréquentes :")
display(pd.crosstab(err["vrai"].map(dict(enumerate(PROFILE_ML_CLASSES))),
                    err["prédit"].map(dict(enumerate(PROFILE_ML_CLASSES))), margins=True))

**Pondération** : la ligne « pondéré WGT » applique les poids d'enquête du SCF. Elle mesure la performance attendue sur la population des ménages américains, et non sur l'échantillon (le SCF sur-représente volontairement les ménages aisés).

**Analyse** : le seuil de décision fixe le compromis entre deux erreurs de gravité différente : classer « Non prudent » un client réellement prudent (risque de lui proposer un portefeuille trop risqué, erreur la plus grave) ou l'inverse (portefeuille trop défensif). La tolérance au risque est en partie psychologique et seulement partiellement expliquée par des variables socio-économiques : dans l'application, la prédiction est donc **proposée** au conseiller, qui peut la corriger (principe de l'humain dans la boucle).

### 4.7 Validation externe sur une autre vague de l'enquête (SCF 2019)
Le test précédent mesure la généralisation à d'autres ménages **de la même enquête** (2022). Pour une validation plus exigeante, on applique le modèle entraîné sur 2022, **sans aucun ré-entraînement ni ajustement du seuil**, aux ménages de l'enquête **2019** : un échantillon indépendant, interrogé avant le COVID et dans un autre contexte de marché.
- Les montants de chaque vague sont exprimés en dollars de l'année d'enquête : ceux de 2019 sont convertis en dollars de 2022 avec l'indice des prix CPI de la FRED.
- Les variables absentes en 2019 sont laissées manquantes et imputées par le pipeline, avec les valeurs apprises sur l'entraînement.

In [ ]:
try:
    scf_ext = dl.download_scf(year=SCF_EXTERNAL_YEAR)
except Exception as exc:
    scf_ext = None
    print("⚠ SCF externe indisponible, validation externe ignorée :", exc)

if scf_ext is not None:
    X_ext, y_ext, g_ext, w_ext, rep_ext = prepare_scf(scf_ext)
    cpi = fred_raw["CPIAUCSL"].dropna()
    infl = cpi.loc[str(SCF_YEAR)].mean() / cpi.loc[str(SCF_EXTERNAL_YEAR)].mean()
    for c in [c for c in MONEY_FEATURES if c in X_ext.columns]:
        X_ext[c] = X_ext[c] * infl
    absentes = [c for c in X_p.columns if c not in X_ext.columns]
    X_ext = X_ext.reindex(columns=X_p.columns)
    y_ext_int = y_ext.astype(str).map(CLS_TO_INT).astype(int)
    print(f"Ménages {SCF_EXTERNAL_YEAR} : {g_ext.nunique()} | facteur d'inflation {SCF_EXTERNAL_YEAR}→{SCF_YEAR} : {infl:.3f}")
    print("Répartition de la cible :", rep_ext["repartition_cible"])
    print("Variables absentes en", SCF_EXTERNAL_YEAR, ":", absentes or "aucune")
    thr = THRESH if BINARY else None
    maj = profile_fitted["Baseline – classe majoritaire"].best_estimator_
    profil_externe = pd.DataFrame({
        f"Test interne – SCF {SCF_YEAR}": eval_multiclass(best_profile, Xp_te, yp_te, thr),
        f"Validation externe – SCF {SCF_EXTERNAL_YEAR}": eval_multiclass(best_profile, X_ext, y_ext_int, thr),
        f"Validation externe – SCF {SCF_EXTERNAL_YEAR} (pondéré WGT)": eval_multiclass(
            best_profile, X_ext, y_ext_int, thr, sample_weight=w_ext.values),
        f"Baseline majoritaire – SCF {SCF_EXTERNAL_YEAR}": eval_multiclass(maj, X_ext, y_ext_int),
    }).T
    profil_externe.to_csv(RESULTS_DIR / "agent_profil_validation_externe.csv")
    RESULTS["profil_externe"] = profil_externe
    display(profil_externe)

**Lecture** : si les performances sur 2019 restent proches de celles du test 2022, le modèle capte une relation **stable** entre situation financière et aversion au risque, et non une particularité de l'échantillon 2022. Une baisse modérée est normale (contexte différent, variables absentes). Une forte baisse signalerait un sur-apprentissage propre à l'enquête 2022.

## 5. Agent Prévision – quels actifs vont surperformer le mois prochain ?

### 5.1 Formulation
Pour chaque actif *i* et date *t* : **cible** $y_{i,t} = \mathbb{1}\{r_{i,t\to t+21} > \text{médiane}_j(r_{j,t\to t+21})\}$.

On prédit une **surperformance relative** plutôt qu'un rendement absolu : le problème est équilibré par construction (≈ 50 %), insensible à la tendance globale du marché, et c'est exactement l'information utile pour choisir **entre** actifs. Le t-stat de l'IC est corrigé par **Newey-West** : les observations hebdomadaires portent sur des rendements à 21 jours qui se chevauchent, ce qui rend le t-stat naïf environ deux fois trop optimiste.

Les probabilités prédites $p_{i,t}$ sont ensuite transformées en rendements espérés excédentaires par la loi fondamentale de la gestion active (Grinold & Kahn) : $\alpha_i = IC \times \sigma_i \times z_i$.

**Variables** (toutes calculées avec l'information disponible en *t*) : momentum (5 j à 12 mois, momentum 12-1), volatilités, drawdown, écarts aux moyennes mobiles, RSI, asymétrie, bêta et corrélation au marché, rangs transversaux, variables macro (VIX, pente de courbe, spread de crédit, inflation, chômage…), classe d'actifs.

In [ ]:
panel = compute_asset_features(prices, macro)
FEATS = ASSET_NUM_FEATURES + [c for c in MACRO_FEATURES if c in panel.columns] + CAT_FEATURES
NUM_F = [c for c in FEATS if c not in CAT_FEATURES]
print("Panel complet :", panel.shape, "| variables :", len(FEATS))
panel.to_csv(DATA_PROC / "panel_features.csv.gz", compression="gzip")
display(panel.tail())

### 5.2 Découpage temporel purgé (anti-fuite)
- **Entraînement** : début → 2019 | **Validation** : 2020-2021 | **Test** : 2022 → fin.
- Échantillonnage hebdomadaire pour limiter la redondance des cibles qui se chevauchent.
- **Purge / embargo** de 35 jours calendaires avant chaque frontière : la cible d'une observation du 20 décembre 2019 dépend des prix de janvier 2020 (validation) ; elle est donc supprimée.
- Aucune permutation aléatoire : on ne prédit jamais le passé avec le futur.

In [ ]:
panel_s = sample_dates(panel).dropna(subset=["target"])
tr, va, te = temporal_split(panel_s)
for nm, d in (("train", tr), ("validation", va), ("test", te)):
    dd = d.index.get_level_values("date")
    print(f"{nm:<11}: {dd.min().date()} → {dd.max().date()} | {len(d):>6} obs | "
          f"{dd.nunique():>4} dates | cible=1 : {d['target'].mean():.1%}")
gap1 = va.index.get_level_values("date").min() - tr.index.get_level_values("date").max()
gap2 = te.index.get_level_values("date").min() - va.index.get_level_values("date").max()
assert gap1.days >= EMBARGO_DAYS and gap2.days >= 1
print(f"Écart train→val : {gap1.days} jours ; val→test : {gap2.days} jours (fin de val déjà purgée)")

X_tr, y_tr, d_tr = tr[FEATS], tr["target"].astype(int), tr.index.get_level_values("date")
X_va, y_va = va[FEATS], va["target"].astype(int)
X_te, y_te = te[FEATS], te["target"].astype(int)

### 5.3 Analyse exploratoire du pouvoir prédictif (entraînement)
Coefficient d'information univarié de chaque variable : corrélation de rang moyenne, date par date, entre la variable et le rendement futur.

In [ ]:
ic_uni = pd.Series({f: information_coefficient(tr[f], tr["fwd_ret"]).mean() for f in NUM_F})
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ic_uni.sort_values().plot.barh(ax=axes[0], color=np.where(ic_uni.sort_values() > 0, "seagreen", "indianred"))
axes[0].set_title("IC univarié moyen (train)")
sns.heatmap(tr[ASSET_NUM_FEATURES].corr(), cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("Corrélations entre variables techniques (train)")
plt.tight_layout()
savefig("10_prevision_ic_univarie")
plt.show()

**Lecture** : les IC univariés sont faibles (typiquement |IC| < 0,05), ce qui est **normal en finance** : les marchés sont proches de l'efficience et un IC de 0,03-0,05 stable est déjà exploitable. Plusieurs variables sont fortement corrélées entre elles (différents horizons de momentum) : les modèles régularisés et les arbres gèrent bien cette redondance.

### 5.4 Baselines et modèles
- **Baseline naïve** : classe majoritaire (AUC = 0,5 par construction).
- **Baseline financière** : règle de momentum « acheter les gagnants des 12 derniers mois » (Jegadeesh & Titman, 1993), sans apprentissage.
- **Modèles** : régression logistique, forêt aléatoire, LightGBM (ou HistGradientBoosting), XGBoost, MLP.
- **Réglage des hyperparamètres** : recherche aléatoire avec **validation croisée temporelle purgée** à fenêtre croissante (`PurgedTimeSeriesSplit`), critère ROC-AUC.

In [ ]:
pre_fc = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("win", Winsorizer(0.005, 0.995)),
                      ("sc", StandardScaler())]), NUM_F),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
])


def eval_forecast(proba: np.ndarray, data: pd.DataFrame) -> dict:
    y = data["target"].astype(int)
    s = pd.Series(proba, index=data.index)
    ic = information_coefficient(s, data["fwd_ret"])
    ls = long_short_spread(s, data["fwd_ret"])
    pred = (proba > 0.5).astype(int)
    return {"Accuracy": accuracy_score(y, pred), "Balanced acc.": balanced_accuracy_score(y, pred),
            "F1": f1_score(y, pred), "ROC-AUC": roc_auc_score(y, proba),
            "IC moyen": ic.mean(), "IC t-stat (Newey-West)": newey_west_tstat(ic),
            "Spread Q5-Q1 (21j)": ls.mean(), "% dates IC>0": (ic > 0).mean()}


def forecast_models():
    m = {
        "Régression logistique": (LogisticRegression(max_iter=3000),
                                  {"clf__C": [0.001, 0.01, 0.1, 1]}),
        "Forêt aléatoire": (RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=RANDOM_STATE),
                            {"clf__max_depth": [4, 6, 10], "clf__min_samples_leaf": [50, 200, 500],
                             "clf__max_features": ["sqrt", 0.3]}),
        "MLP": (MLPClassifier(early_stopping=True, max_iter=300, random_state=RANDOM_STATE),
                {"clf__hidden_layer_sizes": [(32,), (64, 32)], "clf__alpha": [1e-3, 1e-2, 1e-1]}),
    }
    if HAS_LGBM:
        m["LightGBM"] = (lgb.LGBMClassifier(verbose=-1, n_jobs=1, random_state=RANDOM_STATE, subsample_freq=1),
                         {"clf__n_estimators": [200, 400], "clf__learning_rate": [0.01, 0.03, 0.05],
                          "clf__num_leaves": [7, 15, 31], "clf__min_child_samples": [100, 300, 600],
                          "clf__subsample": [0.7, 0.9], "clf__colsample_bytree": [0.5, 0.8],
                          "clf__reg_lambda": [0, 1, 10]})
    else:
        m["HistGradientBoosting"] = (HistGradientBoostingClassifier(random_state=RANDOM_STATE),
                                     {"clf__learning_rate": [0.01, 0.03, 0.05], "clf__max_leaf_nodes": [7, 15, 31],
                                      "clf__min_samples_leaf": [100, 300, 600], "clf__max_iter": [200, 400],
                                      "clf__l2_regularization": [0, 1, 10]})
    if HAS_XGB:
        m["XGBoost"] = (xgb.XGBClassifier(tree_method="hist", eval_metric="logloss", n_jobs=1, random_state=RANDOM_STATE),
                        {"clf__n_estimators": [200, 400], "clf__learning_rate": [0.01, 0.03, 0.05],
                         "clf__max_depth": [2, 3, 4], "clf__min_child_weight": [10, 50, 100],
                         "clf__subsample": [0.7, 0.9], "clf__colsample_bytree": [0.5, 0.8]})
    return m


# --- Baselines (aucun apprentissage) ---
fc_rows = {
    "Baseline – classe majoritaire": eval_forecast(np.full(len(va), 0.5), va),
    "Baseline – momentum 12-1": eval_forecast(va["mom_12_1_rank"].values, va),
}

# --- Modèles appris ---
cv_t = PurgedTimeSeriesSplit(n_splits=4 if not QUICK_RUN else 2)
N_ITER = 12 if not QUICK_RUN else 2
fc_fitted, fc_params = {}, {}
for name, (est, grid) in forecast_models().items():
    t0 = time.time()
    pipe = Pipeline([("pre", pre_fc), ("clf", est)])
    n_comb = int(np.prod([len(v) for v in grid.values()]))
    search = RandomizedSearchCV(pipe, grid, n_iter=min(N_ITER, n_comb), cv=cv_t, scoring="roc_auc",
                                n_jobs=N_JOBS, random_state=RANDOM_STATE, refit=True)
    search.fit(X_tr, y_tr, groups=d_tr)
    fc_fitted[name] = search.best_estimator_
    fc_params[name] = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
    fc_rows[name] = {**eval_forecast(search.best_estimator_.predict_proba(X_va)[:, 1], va),
                     "AUC CV (train)": search.best_score_, "Durée (s)": time.time() - t0}
    print(f"✓ {name:<24} AUC val = {fc_rows[name]['ROC-AUC']:.4f} | IC val = {fc_rows[name]['IC moyen']:.4f}")

fc_comparison = pd.DataFrame(fc_rows).T.sort_values("IC moyen", ascending=False)
fc_comparison.to_csv(RESULTS_DIR / "agent_prevision_comparaison_validation.csv")
RESULTS["prevision_validation"] = fc_comparison
print("Meilleurs hyperparamètres :")
display(pd.Series(fc_params).to_frame("hyperparamètres"))
fc_comparison

**Critère de sélection** : le modèle retenu est celui qui maximise l'**IC moyen sur la validation** (critère directement lié à la valeur économique des prévisions, puisque l'allocation utilise le classement des actifs). L'IC de validation de ce modèle, mesuré hors échantillon, sert ensuite de paramètre `IC` dans $\alpha_i = IC \cdot \sigma_i \cdot z_i$. Si aucun modèle ne bat les baselines, l'IC est ramené à 0 et le système devient une allocation purement fondée sur le risque (garde-fou).

In [ ]:
learned = [n for n in fc_comparison.index if not n.startswith("Baseline")]
best_fc_name = learned[0]
IC_VAL = float(fc_comparison.loc[best_fc_name, "IC moyen"])
best_baseline_ic = fc_comparison.loc[[i for i in fc_comparison.index if i.startswith("Baseline")], "IC moyen"].max()
if IC_VAL <= max(best_baseline_ic, 0):
    print("⚠ Le meilleur modèle ne bat pas les baselines en validation : IC ramené à 0 (allocation fondée sur le risque).")
    IC_USED = 0.0
else:
    # L'IC du meilleur modèle en validation surestime la performance future (biais de sélection,
    # non-stationnarité) : on le réduit de moitié (IC_SHRINK) avant de l'utiliser dans α = IC·σ·z.
    IC_USED = IC_SHRINK * IC_VAL
print(f"Modèle retenu : {best_fc_name} | IC validation = {IC_VAL:.4f} | IC utilisé = {IC_USED:.4f}")

# Ré-entraînement train + validation (frontière déjà purgée) puis évaluation UNIQUE sur le test
trva = pd.concat([tr, va])
best_fc = clone(fc_fitted[best_fc_name]).fit(trva[FEATS], trva["target"].astype(int))
p_te = best_fc.predict_proba(X_te)[:, 1]
fc_test = pd.DataFrame({
    best_fc_name: eval_forecast(p_te, te),
    "Baseline – classe majoritaire": eval_forecast(np.full(len(te), 0.5), te),
    "Baseline – momentum 12-1": eval_forecast(te["mom_12_1_rank"].values, te),
}).T
fc_test.to_csv(RESULTS_DIR / "agent_prevision_test.csv")
RESULTS["prevision_test"] = fc_test
fc_test

In [ ]:
s_te = pd.Series(p_te, index=te.index)
ic_te = information_coefficient(s_te, te["fwd_ret"])
ls_te = long_short_spread(s_te, te["fwd_ret"])
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
ic_te.plot(ax=axes[0], alpha=.35, label="IC hebdomadaire")
ic_te.rolling(13).mean().plot(ax=axes[0], color="k", label="moyenne mobile 3 mois")
axes[0].axhline(0, color="r", lw=.8)
axes[0].legend()
axes[0].set_title("Coefficient d'information sur le test")
(ls_te / 4).cumsum().plot(ax=axes[1], color="seagreen")   # /4 : ~4 observations par horizon de 21 j
axes[1].set_title("Spread Q5-Q1 cumulé (approx., test)")
CalibrationDisplay.from_predictions(y_te, p_te, n_bins=10, strategy="quantile", ax=axes[2], name=best_fc_name)
axes[2].set_title("Courbe de calibration (test)")
plt.tight_layout()
savefig("11_prevision_test")
plt.show()

### 5.5 Interprétabilité de l'agent Prévision et analyse des erreurs

In [ ]:
pi_fc = permutation_importance(best_fc, X_te, y_te, scoring="roc_auc", n_repeats=5 if not QUICK_RUN else 2,
                               random_state=RANDOM_STATE, n_jobs=N_JOBS)
pi_fc = pd.DataFrame({"importance": pi_fc.importances_mean, "écart-type": pi_fc.importances_std},
                     index=FEATS).sort_values("importance")
pi_fc.tail(20).plot.barh(y="importance", xerr="écart-type", legend=False, figsize=(8, 7), color="slateblue")
plt.title("Importance par permutation – agent Prévision (baisse de l'AUC, test)")
savefig("12_prevision_permutation")
plt.show()
TOP_FEATURES = pi_fc.sort_values("importance", ascending=False).index[:10].tolist()
pi_fc.sort_values("importance", ascending=False).to_csv(RESULTS_DIR / "agent_prevision_importance_permutation.csv")

fc_tree = [n for n in fc_comparison.index if n in ("LightGBM", "XGBoost", "Forêt aléatoire", "HistGradientBoosting")]
shap_fc_pipe = best_fc if best_fc_name in fc_tree else fc_fitted[fc_tree[0]]
sv_fc = shap_beeswarm(shap_fc_pipe, X_te, f"SHAP – probabilité de surperformance "
                      f"({best_fc_name if best_fc_name in fc_tree else fc_tree[0]})", "13_prevision_shap", class_idx=1)
if sv_fc is not None:
    top = sv_fc.abs.mean(0).values.argmax()
    plt.figure()
    shap.plots.scatter(sv_fc[:, top], show=False)
    plt.title("Dépendance SHAP de la variable la plus influente")
    savefig("14_prevision_shap_dependance")
    plt.show()

In [ ]:
err_fc = te[["fwd_ret", "target", "asset_class"]].copy()
err_fc["proba"] = p_te
err_fc["année"] = err_fc.index.get_level_values("date").year
by_class = err_fc.groupby("asset_class").apply(lambda g: pd.Series({
    "n": len(g), "AUC": roc_auc_score(g["target"], g["proba"]) if g["target"].nunique() > 1 else np.nan,
    "accuracy": ((g["proba"] > .5) == g["target"]).mean()}))
by_year = err_fc.groupby("année").apply(lambda g: pd.Series({
    "n": len(g), "AUC": roc_auc_score(g["target"], g["proba"]),
    "IC": information_coefficient(g["proba"], g["fwd_ret"]).mean()}))
display(by_class)
display(by_year)
by_class.to_csv(RESULTS_DIR / "agent_prevision_erreurs_par_classe.csv")
by_year.to_csv(RESULTS_DIR / "agent_prevision_erreurs_par_annee.csv")

## 6. Agent Régime de marché – mélange gaussien (non supervisé)

Un **modèle de mélange gaussien** suppose que le vecteur d'état du marché $x_t$ (rendement et log-volatilité du S&P 500 sur 1 mois, drawdown, log du VIX, variation du spread de crédit) est issu de $K$ lois normales multivariées : $p(x) = \sum_k \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k)$, paramètres estimés par l'algorithme **EM**.

**Correction de la version 1** : les variables en niveau (pente de courbe 10 ans – 2 ans, spread de crédit) ne sont pas stationnaires. La courbe des taux s'est inversée de 2022 à 2024, un niveau jamais observé dans la période d'entraînement (2011-2019), et le modèle classait alors **91 % de la période de test en « Stress »**. Le portefeuille réduisait donc son risque en permanence. On n'utilise plus que des variables stationnaires (logarithmes, variations, drawdown) et on contrôle la fréquence des régimes hors échantillon. Chaque composante est interprétée comme un régime. Le modèle est **ajusté sur la période d'entraînement uniquement**, puis appliqué date par date en n'utilisant que l'information disponible (filtrage, pas de lissage rétrospectif).

In [ ]:
mkt_feat = compute_market_features(prices, macro)
reg_cols = [c for c in REGIME_FEATURES if c in mkt_feat]
M_tr = mkt_feat.loc[:TRAIN_END, reg_cols].dropna()
reg_scaler = StandardScaler().fit(M_tr)
Z_tr = reg_scaler.transform(M_tr)

bic = {k: GaussianMixture(k, covariance_type="full", n_init=5, random_state=RANDOM_STATE).fit(Z_tr).bic(Z_tr)
       for k in range(2, 7)}
pd.Series(bic).plot(marker="o", title="Critère BIC selon le nombre de régimes", figsize=(6, 3.5))
savefig("15_regime_bic")
plt.show()

K = 3   # 3 régimes retenus pour l'interprétabilité (Calme / Neutre / Stress) – cf. BIC ci-dessus
gmm = GaussianMixture(K, covariance_type="full", n_init=10, random_state=RANDOM_STATE).fit(Z_tr)
centers = pd.DataFrame(reg_scaler.inverse_transform(gmm.means_), columns=reg_cols)
vol_col = "mkt_logvol_21" if "mkt_logvol_21" in centers else "mkt_vol_21"
order = centers[vol_col].argsort().values
label_map = {int(order[0]): "Calme", int(order[1]): "Neutre", int(order[2]): "Stress"}
centers.index = [label_map[i] for i in range(K)]
display(centers.sort_values(vol_col))

In [ ]:
M_all = mkt_feat[reg_cols].dropna()
reg_series = pd.Series(gmm.predict(reg_scaler.transform(M_all)), index=M_all.index).map(label_map)
colors = {"Calme": "seagreen", "Neutre": "gold", "Stress": "crimson"}
fig, ax = plt.subplots(figsize=(15, 5))
spy = prices["SPY"].reindex(reg_series.index)
ax.plot(spy.index, spy.values, color="k", lw=.8)
for reg, col in colors.items():
    ax.fill_between(spy.index, 0, spy.max() * 1.05, where=(reg_series == reg).values, color=col, alpha=.25, label=reg)
ax.set_ylim(spy.min() * .9, spy.max() * 1.05)
ax.axvline(pd.Timestamp(TRAIN_END), color="blue", ls="--")
ax.text(pd.Timestamp(TRAIN_END), spy.max(), " fin entraînement", color="blue")
ax.legend(loc="upper left")
ax.set_title("Régimes détectés et S&P 500")
savefig("16_regimes")
plt.show()

fwd_spy = prices["SPY"].shift(-21) / prices["SPY"] - 1
oos = reg_series.index > TRAIN_END
reg_eval = pd.DataFrame({"régime": reg_series[oos], "rdt SPY 21j futur": fwd_spy.reindex(reg_series.index)[oos],
                         "vol réalisée 21j future": returns["SPY"].rolling(21).std().shift(-21)
                         .reindex(reg_series.index)[oos] * np.sqrt(252)})
reg_tab = reg_eval.groupby("régime").agg(fréquence=("régime", "size"), **{
    "rdt moyen 21j futur": ("rdt SPY 21j futur", "mean"),
    "vol future": ("vol réalisée 21j future", "mean")})
reg_tab["fréquence"] /= reg_tab["fréquence"].sum()
assert reg_tab["fréquence"].max() < 0.8, "Un régime domine hors échantillon : vérifier la stationnarité des variables"
print("Validation économique hors échantillon (2020 → fin) :")
display(reg_tab)
trans = pd.crosstab(reg_series.shift(1), reg_series, normalize="index")
display(trans.rename_axis("de \\ vers"))
reg_tab.to_csv(RESULTS_DIR / "agent_regime_validation.csv")
RESULTS["regime"] = reg_tab

**Validation** : si le régime « Stress » est suivi d'une volatilité future nettement plus élevée que le régime « Calme » hors échantillon, l'agent apporte une information utile pour réduire le risque. La matrice de transition montre la forte persistance des régimes (diagonale proche de 1), ce qui rend la détection en temps réel exploitable.

## 7. Orchestration multi-agent – démonstration

L'orchestrateur exécute le cycle : **Profil → Régime → Prévision → Risque → Allocation ⇄ Risque (négociation) → Explication**.

**Correction de la version 1 – rendements d'équilibre (Black-Litterman)** : quand les prévisions sont absentes ou faibles, une optimisation moyenne-variance sans rendement espéré revient au portefeuille de **variance minimale**. Les trois profils obtenaient alors presque la même allocation, très obligataire (volatilité réalisée d'environ 5 % pour tous). On part désormais d'une allocation stratégique de référence par profil (`PROFILE_BENCHMARK`), dont on déduit les rendements implicites $\pi = \lambda \Sigma w_{ref}$. Le rendement espéré utilisé est $\mu = \pi + \alpha_{ML}$.

**Propriété exacte** (vérifiée en 7.1) : sans signal ML, avec l'aversion de base (régime Calme), sans contrainte de volatilité active et sans pénalité de rotation, l'optimiseur retrouve **exactement** l'allocation de référence. Ensuite, trois mécanismes l'en écartent volontairement : (1) en régime Neutre ou Stress, l'aversion augmente et la cible de volatilité baisse, ce qui réduit le risque ; (2) si la référence dépasse le budget de volatilité du profil (cas du profil Prudent en 2022), la contrainte la ramène dans ce budget ; (3) la pénalité de rotation freine l'écart au portefeuille précédent. L'agent Prévision incline ensuite le portefeuille autour de ce point. Les agents communiquent via un tableau noir partagé dont on affiche le journal.

In [ ]:
profile_bundle = {"pipeline": best_profile, "num": num_p, "cat": cat_p, "classes": PROFILE_ML_CLASSES,
                  "mode": PROFILE_TARGET_MODE, "threshold": THRESH,
                  "defaults": {**Xp_tr[num_p].median().to_dict(),
                               **{c: Xp_tr[c].mode().iloc[0] for c in cat_p}},
                  "model_name": best_profile_name}
regime_bundle = {"model": gmm, "scaler": reg_scaler, "features": reg_cols,
                 "fill": M_tr.median().to_dict(), "label_map": label_map}
forecast_bundle = {"pipeline": best_fc, "features": FEATS, "ic": IC_USED,
                   "top_features": TOP_FEATURES, "model_name": best_fc_name, "params": fc_params[best_fc_name]}

orchestrator = PortfolioOrchestrator(profile_bundle, regime_bundle, forecast_bundle)

# Clients fictifs : situation financière (variables du SCF, en dollars US) + réponses au questionnaire
# (indice de l'option choisie, de 0 = plus prudent à 3 = plus tolérant ; cf. src/questionnaire.py)
Q = lambda h, o, r, p, e, part, rev: {"Q_HORIZON": h, "Q_OBJECTIF": o, "Q_REACTION": r, "Q_PERTE": p,
                                     "Q_EXPERIENCE": e, "Q_PART": part, "Q_REVENUS": rev}
clients = {
    "Client A – retraité": {"AGE": 68, "KIDS": 0, "INCOME": 45_000, "NETWORTH": 350_000, "ASSET": 380_000,
                            "DEBT": 30_000, "LEVRATIO": 0.08, "DEBT2INC": 0.6, "KNOWL": 5, "FINLIT": 2,
                            "HHSEX": 1, "EDCL": 2, "MARRIED": 1, "OCCAT1": 3, "LIFECL": 5, "HOUSECL": 1, "SAVED": 1,
                            **Q(1, 0, 1, 1, 1, 1, 2)},
    "Client B – cadre 40 ans": {"AGE": 41, "KIDS": 2, "INCOME": 110_000, "NETWORTH": 250_000, "ASSET": 520_000,
                                "DEBT": 270_000, "LEVRATIO": 0.52, "DEBT2INC": 2.4, "KNOWL": 7, "FINLIT": 3,
                                "HHSEX": 1, "EDCL": 4, "MARRIED": 1, "OCCAT1": 1, "LIFECL": 3, "HOUSECL": 1, "SAVED": 1,
                                **Q(2, 2, 2, 2, 2, 1, 2)},
    "Client C – entrepreneur 32 ans": {"AGE": 32, "KIDS": 0, "INCOME": 180_000, "NETWORTH": 600_000,
                                       "ASSET": 700_000, "DEBT": 100_000, "LEVRATIO": 0.14, "DEBT2INC": 0.55,
                                       "KNOWL": 9, "FINLIT": 3, "HHSEX": 1, "EDCL": 4, "MARRIED": 2,
                                       "OCCAT1": 2, "LIFECL": 1, "HOUSECL": 2, "SAVED": 1,
                                       **Q(3, 3, 3, 3, 2, 2, 2)},
    # Cas d'incohérence : se déclare très tolérant, mais situation financière fragile
    "Client D – 26 ans, se déclare dynamique": {"AGE": 26, "KIDS": 1, "INCOME": 22_000, "NETWORTH": -8_000,
                                                "ASSET": 6_000, "DEBT": 14_000, "LEVRATIO": 2.3, "DEBT2INC": 0.64,
                                                "KNOWL": 2, "FINLIT": 1, "HHSEX": 2, "EDCL": 2, "MARRIED": 2,
                                                "OCCAT1": 1, "LIFECL": 1, "HOUSECL": 2, "SAVED": 0,
                                                **Q(3, 3, 3, 3, 1, 3, 2)},
}
last_date = panel.index.get_level_values("date").max()
demo = {}
for name, cl in clients.items():
    res = orchestrator.recommend(last_date, prices, panel, mkt_feat, client=cl)
    demo[name] = res
    print("=" * 90, f"\n{name}\n")
    print(res["explanation"])

profil_tab = pd.DataFrame({n: {"Score questionnaire": f"{r['profile_detail']['questionnaire']['score']}/21",
                               "Tolérance déclarée": r["profile_detail"]["declare"],
                               "P(profil prudent) – ML": f"{r['profile_proba'].get('Prudent', np.nan):.0%}",
                               "Capacité (ML)": r["profile_detail"]["capacite"],
                               "Profil retenu": r["profile"],
                               "Alerte conseiller": "oui" if r["profile_detail"]["alerte"] else "non"}
                           for n, r in demo.items()}).T
display(profil_tab)

In [ ]:
res = demo["Client B – cadre 40 ans"]
print("Journal des échanges entre agents (client B) :")
display(res["log"])
alloc_tab = pd.DataFrame({n: r["weights"] for n, r in demo.items()}).fillna(0)
alloc_tab = alloc_tab[(alloc_tab > 0.005).any(axis=1)]
alloc_tab.insert(0, "Classe", alloc_tab.index.map(ASSET_CLASS))
display(alloc_tab.sort_values("Classe").style.format({c: "{:.1%}" for c in demo}))

fig, axes = plt.subplots(1, len(demo), figsize=(5.5 * len(demo), 5))
for ax, (n, r) in zip(axes, demo.items()):
    ex = pd.Series(r["metrics"]["expositions"])
    ex = ex[ex > 0.005]
    ax.pie(ex, labels=ex.index, autopct="%1.0f%%", startangle=90)
    ax.set_title(f"{n}\nProfil : {r['profile']} | vol {r['metrics']['vol_ex_ante']:.1%}")
plt.tight_layout()
savefig("17_demo_allocations")
plt.show()

### 7.1 Vérification de la propriété de Black-Litterman
Pour plusieurs dates et chaque profil, on optimise avec $\mu = \pi$ (sans signal ML) : (a) dans les conditions de la propriété (aversion de base, cible de volatilité non contraignante, pas de portefeuille précédent), puis (b) en régime Stress. On mesure l'écart maximal aux poids de référence, en points de pourcentage.

In [ ]:
bl_rows = []
for d in [pd.Timestamp("2019-06-28"), pd.Timestamp("2022-06-30"), last_date]:
    d = prices.index[prices.index <= d][-1]
    R = returns.loc[:d].tail(LOOKBACK_COV)
    R = R.loc[:, R.notna().mean() >= 0.9]
    cov_d = RiskAgent().estimate(R)
    for prof in PROFILE_CLASSES:
        c = PROFILE_CONSTRAINTS[prof]
        w_ref = PortfolioOrchestrator.benchmark_weights(prof, list(cov_d.index))
        pi = pd.Series(c["risk_aversion"] * cov_d.values @ w_ref.values, index=cov_d.index)
        w_a = AllocationAgent().optimize(pi, cov_d, c, 10.0, c["risk_aversion"], None)
        m_s = REGIME_VOL_MULTIPLIER["Stress"]
        w_b = AllocationAgent().optimize(pi, cov_d, c, c["target_vol"] * m_s, c["risk_aversion"] / m_s, None)
        bl_rows.append({"date": d.date(), "profil": prof,
                        "vol. de la référence": float(np.sqrt(w_ref @ cov_d @ w_ref)),
                        "cible de vol.": c["target_vol"],
                        "écart max (a) propriété": (w_a - w_ref).abs().max(),
                        "écart max (b) régime Stress": (w_b - w_ref).abs().max()})
bl_check = pd.DataFrame(bl_rows).set_index(["date", "profil"])
bl_check.to_csv(RESULTS_DIR / "verification_black_litterman.csv")
assert bl_check["écart max (a) propriété"].max() < 0.005, "La propriété de Black-Litterman n'est pas vérifiée"
display(bl_check.style.format("{:.2%}"))

## 8. Backtest walk-forward sur la période de test

**Protocole** (aucune information future) :
- rebalancement **mensuel** (dernier jour de bourse du mois) sur toute la période de test ;
- le modèle de prévision est **ré-entraîné chaque 1er janvier** sur toutes les observations dont la cible est connue à cette date (fenêtre croissante + embargo), avec les hyperparamètres choisis en validation ;
- covariance estimée sur les 252 derniers jours ;
- coûts de transaction de 10 points de base sur chaque euro échangé ;
- la pénalité de rotation γ et l'intensité de la réduction du risque par régime sont choisies sur la **période de validation** (2020-2021), jamais sur le test.

**Stratégies comparées** :
| Stratégie | Type |
|---|---|
| Équipondérée 1/N | Baseline simple (DeMiguel et al., 2009 : difficile à battre) |
| 60/40 (S&P 500 / Trésor 7-10 ans) | Référence de l'industrie |
| Variance minimale | Optimisation fondée sur le risque |
| Markowitz historique | Optimisation moyenne-variance classique (rendements = moyennes passées) |
| Multi-agent **sans ML** (α = 0) | **Ablation** : mêmes contraintes et régime, sans l'agent Prévision |
| **Référence stratégique** – par profil | Allocation de référence du profil, rebalancée chaque mois, sans optimisation ni régime |
| **1/N à volatilité cible** – par profil | 1/N combiné à des bons du Trésor 1-3 ans pour atteindre la volatilité cible du profil : **même niveau de risque** |
| **Multi-agent complet** | Système proposé, pour chaque profil |

Le 1/N simple a une volatilité d'environ 12 % : le comparer directement au profil Prudent (cible 6 %) n'est pas équitable. Les deux nouvelles références par profil permettent une comparaison **à risque comparable**.

### 8.0 Réglages choisis sur la validation : pénalité de rotation et réduction du risque par régime
Deux réglages de l'allocation ne sont pas appris par un modèle. Ils sont choisis **sur la période de validation 2020-2021, jamais sur le test**, avec le modèle de prévision entraîné sur la seule période d'entraînement. Le critère est le ratio de Sharpe moyen des trois profils.
- **γ, la pénalité de rotation** (‖w − w_précédent‖²) : sans frein, l'optimiseur réalloue fortement le portefeuille chaque mois, et les coûts de transaction absorbent le gain.
- **L'intensité de la réduction du risque par régime** : de « aucune » à « forte » (en régime Stress, cible de volatilité × 1,00 / 0,90 / 0,75 / 0,60). Un premier diagnostic a montré que réduire le risque en régime Stress protège contre les pertes, mais peut coûter du rendement : les périodes agitées ont souvent été suivies de rebonds. Ce compromis doit être arbitré sur des données qui ne servent pas au test.

In [ ]:
val_reb = [d for d in rebalance_dates(prices.index, pd.Timestamp(TRAIN_END) - pd.Timedelta(days=7),
                                      pd.Timestamp(VAL_END) - pd.Timedelta(days=EMBARGO_DAYS))
           if d in set(panel.index.get_level_values("date"))]
val_bundle = {**forecast_bundle, "pipeline": fc_fitted[best_fc_name]}   # modèle entraîné sur le train seul
gammas = [0.5, 2.0, 5.0, 10.0] if not QUICK_RUN else [TURNOVER_PENALTY]
mult_grid = REGIME_MULTIPLIER_GRID if not QUICK_RUN else {"modérée": REGIME_MULTIPLIER_GRID["modérée"],
                                                          "aucune": REGIME_MULTIPLIER_GRID["aucune"]}
cal_rows = {}
t0 = time.time()
for mult_name, mult in mult_grid.items():
    for gamma in gammas:
        for prof in PROFILE_CLASSES:
            Wv, prev_v = {}, None
            orch_v = PortfolioOrchestrator(None, {**regime_bundle, "vol_multiplier": mult},
                                           {**val_bundle, "turnover_penalty": gamma})
            for d in val_reb:
                r = orch_v.recommend(d, prices, panel, mkt_feat, profile=prof, prev_w=prev_v, explain=False)
                Wv[d], prev_v = r["weights"], r["weights"]
            rv, to_v = simulate(Wv, returns.loc[:VAL_END])
            m = perf_metrics(rv, rf_daily, to_v)
            cal_rows[(mult_name, gamma, prof)] = {"Sharpe": m["Sharpe"], "Max drawdown": m["Max drawdown"],
                                                  "Rotation annuelle": m["Rotation annuelle"]}
print(f"Grille évaluée en {time.time() - t0:.0f} s")
cal_tab = pd.DataFrame(cal_rows).T
cal_tab.index.names = ["réduction du risque", "γ", "profil"]
cal_summary = cal_tab.groupby(level=["réduction du risque", "γ"]).mean()
best_mult_name, TURNOVER_GAMMA = cal_summary["Sharpe"].idxmax()
TURNOVER_GAMMA = float(TURNOVER_GAMMA)
regime_bundle["vol_multiplier"] = mult_grid[best_mult_name]
regime_bundle["vol_multiplier_name"] = best_mult_name
forecast_bundle["turnover_penalty"] = TURNOVER_GAMMA
cal_tab.to_csv(RESULTS_DIR / "choix_reglages_validation.csv")
RESULTS["reglages_validation"] = cal_summary
display(cal_summary.sort_values("Sharpe", ascending=False).style.format(
    {"Sharpe": "{:.3f}", "Max drawdown": "{:.2%}", "Rotation annuelle": "{:.2f}"}))
print(f"Réglages retenus (validation) : réduction du risque « {best_mult_name} » "
      f"{mult_grid[best_mult_name]} ; γ = {TURNOVER_GAMMA}")

In [ ]:
test_start = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
reb = rebalance_dates(prices.index, test_start - pd.Timedelta(days=7), panel.index.get_level_values("date").max())
panel_dates = set(panel.index.get_level_values("date"))
reb = [d for d in reb if d in panel_dates]
print(f"{len(reb)} dates de rebalancement : {reb[0].date()} → {reb[-1].date()}")

emb = pd.Timedelta(days=EMBARGO_DAYS)
yearly_models = {}


def forecast_bundle_at(date):
    """Modèle de prévision ré-entraîné annuellement avec les seules cibles connues à cette date."""
    year = date.year
    if year not in yearly_models:
        cutoff = pd.Timestamp(f"{year}-01-01")
        d_all = panel_s.index.get_level_values("date")
        train_wf = panel_s[d_all <= cutoff - emb]
        mdl = clone(fc_fitted[best_fc_name]).fit(train_wf[FEATS], train_wf["target"].astype(int))
        yearly_models[year] = {**forecast_bundle, "pipeline": mdl}
        print(f"  modèle {year} entraîné jusqu'au {(cutoff - emb).date()} ({len(train_wf)} obs)")
    return yearly_models[year]


W = {}
for prof in PROFILE_CLASSES:
    for variant in ("Multi-agent", "Multi-agent sans ML"):
        W[f"{variant} – {prof}"] = {}
prev = {k: None for k in W}
t0 = time.time()
for d in reb:
    fb = forecast_bundle_at(d)
    for prof in PROFILE_CLASSES:
        for variant, bundle in (("Multi-agent", fb), ("Multi-agent sans ML", {**fb, "ic": 0.0})):
            key = f"{variant} – {prof}"
            orch = PortfolioOrchestrator(None, regime_bundle, bundle)
            r = orch.recommend(d, prices, panel, mkt_feat, profile=prof, prev_w=prev[key], explain=False)
            W[key][d] = r["weights"]
            prev[key] = r["weights"]
print(f"Allocations multi-agent calculées en {time.time() - t0:.0f} s")

for d in reb:
    window = returns.loc[:d].tail(LOOKBACK_COV)
    avail = window.columns[window.notna().mean() >= 0.9].tolist()
    W.setdefault("Équipondérée 1/N", {})[d] = equal_weight(avail)
    W.setdefault("60/40", {})[d] = sixty_forty(avail)
    W.setdefault("Variance minimale", {})[d] = min_variance(window[avail])
    W.setdefault("Markowitz historique", {})[d] = markowitz_historical(window[avail])
    for prof in PROFILE_CLASSES:
        W.setdefault(f"Référence stratégique – {prof}", {})[d] = PortfolioOrchestrator.benchmark_weights(prof, avail)
        W.setdefault(f"1/N vol. cible – {prof}", {})[d] = vol_targeted_equal_weight(
            window[avail], PROFILE_CONSTRAINTS[prof]["target_vol"])

In [ ]:
bt_returns, bt_rows = {}, {}
for name, wd in W.items():
    r, turnover = simulate(wd, returns)
    bt_returns[name] = r
    bt_rows[name] = perf_metrics(r, rf_daily, turnover)
bt_returns = pd.DataFrame(bt_returns)
bt_table = pd.DataFrame(bt_rows).T
bt_table.to_csv(RESULTS_DIR / "backtest_performances_test.csv")
bt_returns.to_csv(RESULTS_DIR / "backtest_rendements_quotidiens.csv")
RESULTS["backtest"] = bt_table
fmt = ({c: "{:.2%}" for c in bt_table.columns if c not in ("Sharpe", "Sortino", "Calmar", "Rotation annuelle")}
       | {c: "{:.2f}" for c in ("Sharpe", "Sortino", "Calmar", "Rotation annuelle")})
display(bt_table.style.format(fmt).background_gradient(subset=["Sharpe"], cmap="RdYlGn"))

# Comparaison à risque comparable, profil par profil
cols_rc = ["Rendement annualisé", "Volatilité annualisée", "Sharpe", "Max drawdown", "Rotation annuelle"]
rc = []
for prof in PROFILE_CLASSES:
    for strat in (f"Multi-agent – {prof}", f"Multi-agent sans ML – {prof}",
                  f"Référence stratégique – {prof}", f"1/N vol. cible – {prof}"):
        rc.append(bt_table.loc[strat, cols_rc].rename((prof, strat.split(" – ")[0])))
risk_comp = pd.DataFrame(rc)
risk_comp.index = pd.MultiIndex.from_tuples(risk_comp.index, names=["profil", "stratégie"])
risk_comp.to_csv(RESULTS_DIR / "backtest_risque_comparable.csv")
RESULTS["risque_comparable"] = risk_comp
display(risk_comp.style.format({k: v for k, v in fmt.items() if k in risk_comp.columns}))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
wealth = (1 + bt_returns.fillna(0)).cumprod()
bench = ["Équipondérée 1/N", "60/40", "Variance minimale", "Markowitz historique"]
for c in bench:
    wealth[c].plot(ax=axes[0], ls="--", lw=1.2)
for c in [f"Multi-agent – {p}" for p in PROFILE_CLASSES]:
    wealth[c].plot(ax=axes[0], lw=2)
axes[0].set_title("Valeur du portefeuille (base 1) – période de test")
axes[0].legend(ncol=2)
dd = wealth / wealth.cummax() - 1
for c in bench[:2] + [f"Multi-agent – {p}" for p in PROFILE_CLASSES]:
    dd[c].plot(ax=axes[1], lw=1.2)
axes[1].set_title("Drawdowns")
plt.tight_layout()
savefig("18_backtest_valeur_drawdown")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharey=False)
for ax, prof in zip(axes, PROFILE_CLASSES):
    for strat, style in ((f"Multi-agent – {prof}", "-"), (f"Multi-agent sans ML – {prof}", "-."),
                         (f"Référence stratégique – {prof}", ":"), (f"1/N vol. cible – {prof}", "--")):
        wealth[strat].plot(ax=ax, ls=style, lw=1.6, label=strat.split(" – ")[0])
    ax.set_title(f"Profil {prof} – à risque comparable")
    ax.legend(fontsize=8)
plt.tight_layout()
savefig("18b_backtest_risque_comparable")
plt.show()

In [ ]:
wq = pd.DataFrame(W["Multi-agent – Équilibré"]).T.fillna(0)
wq_cls = wq.T.groupby(wq.columns.map(ASSET_CLASS)).sum().T
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
wq_cls.plot.area(ax=axes[0], linewidth=0)
axes[0].set_title("Allocation par classe – Multi-agent Équilibré")
axes[0].set_ylim(0, 1)
reg_at = reg_series.reindex(wq.index)
axes[1].bar(range(len(reg_at)), 1, color=reg_at.map(colors), width=1)
axes[1].set_xticks(range(0, len(reg_at), 6))
axes[1].set_xticklabels([d.strftime("%Y-%m") for d in reg_at.index[::6]], rotation=45)
axes[1].set_yticks([])
axes[1].set_title("Régime détecté à chaque rebalancement (vert=Calme, jaune=Neutre, rouge=Stress)")
plt.tight_layout()
savefig("19_backtest_allocations")
plt.show()

### 8.1 Significativité statistique
Les écarts de ratio de Sharpe sur quelques années sont très bruités. Comme dans le tableau de performances, le Sharpe est calculé sur les rendements **excédentaires** (diminués du taux sans risque). On estime un intervalle de confiance de la différence de Sharpe entre le système complet et son ablation (sans ML), ainsi qu'avec la stratégie 1/N, par **bootstrap par blocs** (blocs de 21 jours pour préserver l'autocorrélation et le regroupement de volatilité).

In [ ]:
def block_bootstrap_sharpe_diff(r1, r2, n_boot=2000 if not QUICK_RUN else 200, block=21, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    pair = pd.concat([r1, r2], axis=1).dropna()
    # Rendements excédentaires (même convention que perf_metrics)
    df = pair.sub(rf_daily.reindex(pair.index).fillna(0), axis=0).values
    n = len(df)
    nb = int(np.ceil(n / block))
    diffs = []
    for _ in range(n_boot):
        starts = rng.integers(0, n - block, nb)
        idx = np.concatenate([np.arange(s, s + block) for s in starts])[:n]
        s = df[idx]
        sh = s.mean(0) / s.std(0) * np.sqrt(252)
        diffs.append(sh[0] - sh[1])
    diffs = np.array(diffs)
    return {"diff. Sharpe observée": (df.mean(0) / df.std(0) * np.sqrt(252) @ np.array([1, -1])),
            "IC 95% bas": np.quantile(diffs, .025), "IC 95% haut": np.quantile(diffs, .975),
            "P(diff ≤ 0)": (diffs <= 0).mean()}


sig = {}
for p in PROFILE_CLASSES:
    sig[f"{p} : complet vs sans ML"] = block_bootstrap_sharpe_diff(bt_returns[f"Multi-agent – {p}"],
                                                                   bt_returns[f"Multi-agent sans ML – {p}"])
    sig[f"{p} : complet vs référence stratégique"] = block_bootstrap_sharpe_diff(
        bt_returns[f"Multi-agent – {p}"], bt_returns[f"Référence stratégique – {p}"])
    sig[f"{p} : complet vs 1/N vol. cible"] = block_bootstrap_sharpe_diff(
        bt_returns[f"Multi-agent – {p}"], bt_returns[f"1/N vol. cible – {p}"])
sig = pd.DataFrame(sig).T
sig.to_csv(RESULTS_DIR / "backtest_significativite.csv")
RESULTS["significativite"] = sig
sig

**Interprétation à rédiger après exécution** : comparer les profils entre eux (le risque réalisé doit croître de Prudent à Dynamique, ce qui valide l'adéquation aux profils), comparer au 1/N et au 60/40, et lire l'ablation « sans ML » pour isoler la valeur ajoutée de l'agent Prévision. Si l'intervalle de confiance de la différence de Sharpe contient 0, la surperformance n'est pas statistiquement établie sur cette période : c'est une conclusion honnête et attendue sur 3-4 ans de données.

## 9. Sauvegarde des artefacts pour le déploiement

In [ ]:
joblib.dump(profile_bundle, MODELS_DIR / "agent_profil.joblib")
joblib.dump(regime_bundle, MODELS_DIR / "agent_regime.joblib")
final_fc_bundle = forecast_bundle_at(last_date + pd.Timedelta(days=366))   # modèle avec toutes les cibles connues
joblib.dump(final_fc_bundle, MODELS_DIR / "agent_prevision.joblib")

recent_dates = panel.index.get_level_values("date").unique().sort_values()[-260:]
panel.loc[panel.index.get_level_values("date").isin(recent_dates)].to_csv(DATA_PROC / "panel_recent.csv")
mkt_feat.to_csv(DATA_PROC / "market_features.csv")
reg_series.rename("regime").to_csv(DATA_PROC / "regimes.csv")
meta = {"date_donnees": str(last_date.date()), "modele_profil": best_profile_name,
        "modele_prevision": best_fc_name, "IC_validation": IC_VAL, "IC_utilise": IC_USED,
        "penalite_rotation": TURNOVER_GAMMA, "reduction_risque_regime": best_mult_name,
        "variables_prevision": FEATS, "top_variables": TOP_FEATURES,
        "lightgbm": HAS_LGBM, "xgboost": HAS_XGB, "shap": HAS_SHAP}
(MODELS_DIR / "metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("Artefacts enregistrés :", sorted(p.name for p in MODELS_DIR.iterdir()))

## 10. Synthèse automatique des résultats

La cellule suivante génère `results/resultats_synthese.md`, qui regroupe tous les tableaux de performances à reporter dans le rapport et la présentation.

In [ ]:
def to_md(df: pd.DataFrame) -> str:
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].map(lambda v: f"{v:.4f}")
    try:
        return df.to_markdown()
    except ImportError:
        return "```\n" + df.to_string() + "\n```"


sections = {
    "Agent Profil – comparaison des modèles (validation)": RESULTS["profil_validation"],
    "Agent Profil – modèle retenu vs baselines (test)": RESULTS["profil_test"],
    **({"Agent Profil – validation externe (autre vague SCF)": RESULTS["profil_externe"]}
       if "profil_externe" in RESULTS else {}),
    "Agent Prévision – comparaison des modèles (validation)": RESULTS["prevision_validation"],
    "Agent Prévision – modèle retenu vs baselines (test)": RESULTS["prevision_test"],
    "Agent Régime – validation hors échantillon": RESULTS["regime"],
    "Réglages de l'allocation choisis sur la validation (moyenne des 3 profils)": RESULTS["reglages_validation"],
    "Backtest – performances sur la période de test": RESULTS["backtest"],
    "Backtest – comparaison à risque comparable": RESULTS["risque_comparable"],
    "Backtest – significativité (bootstrap par blocs)": RESULTS["significativite"],
}
md = [f"# Synthèse des résultats\n\nDonnées jusqu'au {last_date.date()} – "
      f"modèle Profil : **{best_profile_name}** – modèle Prévision : **{best_fc_name}** "
      f"(IC validation = {IC_VAL:.4f}).\n"]
for title, df in sections.items():
    md.append(f"\n## {title}\n\n{to_md(df)}\n")
(RESULTS_DIR / "resultats_synthese.md").write_text("\n".join(md), encoding="utf-8")
print("\n".join(md)[:3000])

## Conclusion, limites et perspectives

**Ce que le système apporte** : une chaîne complète et automatisée – données réelles → profil client → lecture du régime de marché → prévisions → allocation sous contraintes → contrôle des risques → explication – évaluée sans fuite de données, avec des baselines et une ablation.

**Limites**
- **Efficience des marchés** : le pouvoir prédictif d'un mois sur l'autre est faible (IC de quelques centièmes) ; les résultats d'un backtest de 3-4 ans restent statistiquement fragiles.
- **Non-stationnarité** : les relations apprises (ex. corrélation actions/obligations) changent ; d'où le ré-entraînement annuel et le contrôle par régime.
- **Données SCF américaines** : le modèle de profil reflète des ménages américains ; pour un déploiement au Maroc ou en Europe, il faudrait le recalibrer avec un questionnaire local (conforme MiFID II / réglementation AMMC).
- **Univers d'ETF américains** cotés en dollars ; pas de prise en compte du change ni de la fiscalité.
- Les coûts de transaction sont simplifiés (coût proportionnel fixe).

**Perspectives**
- Agent d'apprentissage par renforcement (environnement `Gymnasium`, algorithme PPO) pour apprendre directement la politique d'allocation.
- Modèle de Black-Litterman pour combiner l'équilibre de marché et les vues de l'agent Prévision.
- Agent conversationnel (grand modèle de langage) pour dialoguer avec le client et reformuler les explications.
- Données alternatives (sentiment de presse via modèles Hugging Face de type FinBERT).

> *Ce projet est réalisé à des fins pédagogiques et ne constitue pas un conseil en investissement.*